In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from google.colab import drive
drive.mount('/content/drive')

# 1. Đọc dữ liệu và tạo đặc trưng cơ bản
df = pd.read_csv('/content/drive/MyDrive/Project ML/Models/processed_bank_data.csv')

df.head()

Mounted at /content/drive


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,age_group_adult,age_group_middle,age_group_senior
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191,...,False,True,False,False,False,True,False,False,True,False
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191,...,False,True,False,False,False,True,False,False,True,False
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191,...,False,True,False,False,False,True,False,True,False,False
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191,...,False,True,False,False,False,True,False,True,False,False
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191,...,False,True,False,False,False,True,False,False,True,False


In [ ]:
df['is_contacted_before'] = (df['pdays'] != 999).astype(int)

X = df.drop(columns=['y'])
y = df['y']

# 2. Time Split: Không xáo trộn dữ liệu (shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Tạo 2 kịch bản
X_train_with_dur = X_train.copy()
X_test_with_dur = X_test.copy()

X_train_no_dur = X_train.drop(columns=['duration'])
X_test_no_dur = X_test.drop(columns=['duration'])

# 4. Hàm tiền xử lý và SMOTE
def process_scenario(X_tr, X_te, y_tr):
    # Ép kiểu toàn bộ DataFrame về dạng số float (để chuyển đổi các cột True/False nếu có)
    X_tr = X_tr.astype(float)
    X_te = X_te.astype(float)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    # Chuyển lại mảng numpy thành DataFrame để giữ tên cột
    feature_names = X_tr.columns
    X_tr_processed = pd.DataFrame(X_tr_scaled, columns=feature_names)
    X_te_processed = pd.DataFrame(X_te_scaled, columns=feature_names)

    # SMOTE chỉ áp dụng trên tập Train
    smote = SMOTE(random_state=42)
    X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr_processed, y_tr)

    return X_tr_resampled, y_tr_resampled, X_te_processed

# 5. Thực thi cho từng kịch bản
# Kịch bản 1: Giữ nguyên duration làm benchmark
X_train_res_1, y_train_res_1, X_test_proc_1 = process_scenario(X_train_with_dur, X_test_with_dur, y_train)

# Kịch bản 2: Bỏ duration để mô phỏng thực tế
X_train_res_2, y_train_res_2, X_test_proc_2 = process_scenario(X_train_no_dur, X_test_no_dur, y_train)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt

# Hàm huấn luyện, đánh giá và giải thích mô hình
def train_and_evaluate_lr(X_train, y_train, X_test, y_test, scenario_name):
    print(f"==================================================")
    print(f"BÁO CÁO MÔ HÌNH: {scenario_name}")
    print(f"==================================================")

    # 1. Khởi tạo và huấn luyện Logistic Regression
    # Tăng max_iter để đảm bảo thuật toán hội tụ với dữ liệu lớn
    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(X_train, y_train)

    # 2. Dự đoán trên tập Test
    y_pred = lr_model.predict(X_test)
    y_pred_proba = lr_model.predict_proba(X_test)[:, 1] # Xác suất khách hàng đồng ý (nhãn 1)

    # 3. Đánh giá hiệu suất
    print("\n1. ĐÁNH GIÁ HIỆU SUẤT (CLASSIFICATION REPORT & AUC):")
    print(classification_report(y_test, y_pred, target_names=['Từ chối (0)', 'Đồng ý (1)']))
    auc_score = roc_auc_score(y_test, y_pred_proba)
    print(f"ROC-AUC Score: {auc_score:.4f}")

    # 4. Trích xuất trọng số để giải thích (Business Insight)
    print("\n2. GIẢI THÍCH MÔ HÌNH (TOP ĐẶC TRƯNG QUAN TRỌNG NHẤT):")

    # Lấy tên cột và trọng số tương ứng
    feature_names = X_train.columns
    coefficients = lr_model.coef_[0]

    # Tạo DataFrame để dễ quan sát
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients
    })

    # Sắp xếp các biến theo mức độ tác động
    # Tác động tích cực nhất (Làm tăng xác suất chốt sale)
    top_positive = feature_importance.sort_values(by='Coefficient', ascending=False).head(5)
    # Tác động tiêu cực nhất (Làm giảm xác suất chốt sale)
    top_negative = feature_importance.sort_values(by='Coefficient', ascending=True).head(5)

    print("\n  (+) Top 5 yếu tố làm TĂNG khả năng khách hàng chốt sale:")
    for idx, row in top_positive.iterrows():
        print(f"      - {row['Feature']}: {row['Coefficient']:.4f}")

    print("\n  (-) Top 5 yếu tố làm GIẢM khả năng khách hàng chốt sale:")
    for idx, row in top_negative.iterrows():
        print(f"      - {row['Feature']}: {row['Coefficient']:.4f}")

    print("\n")
    return lr_model

# --- THỰC THI MÔ HÌNH ---

# Kịch bản 1: Có dùng Duration (Benchmark)
model_1 = train_and_evaluate_lr(X_train_res_1, y_train_res_1, X_test_proc_1, y_test,
                                "KỊCH BẢN 1 - CÓ DÙNG DURATION (BENCHMARK)")

# Kịch bản 2: Không dùng Duration (Mô phỏng thực tế)
model_2 = train_and_evaluate_lr(X_train_res_2, y_train_res_2, X_test_proc_2, y_test,
                                "KỊCH BẢN 2 - KHÔNG DÙNG DURATION (THỰC TẾ)")

BÁO CÁO MÔ HÌNH: KỊCH BẢN 1 - CÓ DÙNG DURATION (BENCHMARK)

1. ĐÁNH GIÁ HIỆU SUẤT (CLASSIFICATION REPORT & AUC):
              precision    recall  f1-score   support

 Từ chối (0)       0.77      0.85      0.81      5698
  Đồng ý (1)       0.57      0.44      0.50      2540

    accuracy                           0.73      8238
   macro avg       0.67      0.65      0.65      8238
weighted avg       0.71      0.73      0.71      8238

ROC-AUC Score: 0.7135

2. GIẢI THÍCH MÔ HÌNH (TOP ĐẶC TRƯNG QUAN TRỌNG NHẤT):

  (+) Top 5 yếu tố làm TĂNG khả năng khách hàng chốt sale:
      - euribor3m: 4.8970
      - duration: 2.3721
      - education_university.degree: 0.2453
      - education_professional.course: 0.0953
      - poutcome_success: 0.0852

  (-) Top 5 yếu tố làm GIẢM khả năng khách hàng chốt sale:
      - nr.employed: -2.0428
      - emp.var.rate: -1.9213
      - cons.conf.idx: -1.5063
      - cons.price.idx: -1.1177
      - month_nov: -1.0125


BÁO CÁO MÔ HÌNH: KỊCH BẢN 2 - KHÔNG D

1. Đánh giá Hiệu suất
* Sự thật về biến duration: Kịch bản 1 có chỉ số ROC-AUC (0.7135) cao hơn hẳn Kịch bản 2 (0.6417). Điều này chứng minh duration chứa "thông tin rò rỉ" từ tương lai (gọi càng lâu khách càng dễ chốt). Trong thực tế, Kịch bản 2 mới là thước đo chính xác nhất về năng lực của mô hình trước khi telesale nhấc máy.

* Tác động của Time Split: ROC-AUC ở mức 0.64 là khá thấp so với lúc shuffle 0.78. Mô hình học từ quá khứ (Train) nhưng áp dụng cho tương lai (Test) với bối cảnh kinh tế thay đổi (Time Drift). Điều này cho thấy dữ liệu rất nhạy cảm với thời gian.

* Sự đánh đổi Precision - Recall (Độ chính xác vs. Độ phủ): Ở Kịch bản thực tế (KB2), nhờ thuật toán SMOTE, Recall của lớp Đồng ý đạt 65% (nghĩa là mô hình đã "vớt" được 65% tổng số khách hàng thực sự có nhu cầu). Đổi lại, mô hình phải "hy sinh" Precision (chỉ đạt 39%), nghĩa là trong số những người mô hình bảo nên gọi, chỉ có 39% là đồng ý thật. Đây là sự đánh đổi có thể chấp nhận vì: Thà gọi nhầm còn hơn bỏ sót.

2. Phương hướng phát triển tiếp theo

* Chuyển sang mô hình Dạng cây: Random Forest, XGBoost hoặc LightGBM. Vì các mô hình này có khả năng "bắt" được các quy luật phi tuyến (chữ U, ziczac) mà Logistic Regression bỏ lỡ. Đặc biệt, chúng không hề hấn gì trước hiện tượng đa cộng tuyến của nhóm biến Vĩ mô.

* Tối ưu hóa siêu tham số: Khi huấn luyện Random Forest / XGBoost, hãy sử dụng GridSearchCV hoặc RandomizedSearchCV (chia theo TimeSeriesSplit) để tìm bộ tham số giúp cân bằng tốt nhất giữa ROC-AUC và F1-Score.